In [1]:
a =5 

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
import requests
import json

In [4]:
class State(TypedDict):
    question:str
    products:list
    answer:str

In [5]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

In [6]:
def question_node(state: State):

    print("Question Node Running...")

    return {
        "question": state["question"]
    }

In [7]:
API_URL = "https://6a96580bfa33b37f821b284e.mockapi.io/users"

In [8]:
def api_node(state: State):

    print("API Node Running...")

    response = requests.get(API_URL)

    if response.status_code != 200:
        raise Exception(
            f"API Error: {response.status_code}"
        )

    data = response.json()

    print(f"Fetched {len(data)} records")

    return {
        "products": data
    }

In [9]:
response = requests.get(API_URL)

print("Status Code:", response.status_code)

data = response.json()

print("Total Records:", len(data))

print(data[:3])

Status Code: 200
Total Records: 82
[{'id': 1, 'name': 'Laptop', 'price': 22}, {'id': 2, 'name': 'Smartphone', 'price': 44}, {'id': 3, 'name': 'Tablet', 'price': 92}]


In [10]:
def llm_node(state: State):

    print("LLM Node Running...")

    question = state["question"]
    products = state["products"]

    product_data = json.dumps(
        products,
        indent=2
    )

    prompt = f"""
You are a product data assistant.

USER QUESTION:
{question}

PRODUCT DATA FROM API:
{product_data}

Answer the user's question using ONLY the product data provided above.

Rules:
- Do not invent any product.
- Do not invent any price.
- Carefully examine all products.
- If the user asks for the highest price, find the product with the highest price.
- If the user asks for the lowest price, find the product with the lowest price.
- If the user asks for top products, provide them in descending price order.
- Keep the answer clear and structured.

Use this format when appropriate:

Answer:
<direct answer>

Product:
<product name>

Price:
<price>

ID:
<id>

Explanation:
<short explanation>
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

In [11]:
def response_node(state: State):

    print("\nResponse Node Running...")

    print("\n-----------------------------")
    print("USER QUESTION")
    print("-----------------------------")
    print(state["question"])

    print("\n-----------------------------")
    print("FINAL ANSWER")
    print("-----------------------------")
    print(state["answer"])

    return {
        "answer": state["answer"]
    }

In [12]:
graph = StateGraph(State)

In [13]:
graph.add_node("question", question_node)

graph.add_node("api", api_node)

graph.add_node("llm", llm_node)

graph.add_node("response", response_node)

In [14]:
graph.add_edge(START, "question")

graph.add_edge("question", "api")

graph.add_edge("api", "llm")

graph.add_edge("llm", "response")

graph.add_edge("response", END)

In [15]:
app = graph.compile()

In [16]:
user_question = input("Ask your question: ")

In [17]:
result = app.invoke({
    "question": user_question,
    "products": [],
    "answer": ""
})

Question Node Running...
API Node Running...
Fetched 82 records
LLM Node Running...

Response Node Running...

-----------------------------
USER QUESTION
-----------------------------
which product has highest price?

-----------------------------
FINAL ANSWER
-----------------------------
[{'type': 'text', 'text': 'Answer:\nThe product with the highest price is the Projector.\n\nProduct:\nProjector\n\nPrice:\n98\n\nID:\n26\n\nExplanation:\nAfter reviewing the provided product data, the Projector (ID: 26) has the highest price at 98.', 'extras': {'signature': 'EpANCo0NARFNMg+QyC7X96pDtbhUukYtY35dDpnTxw+2+nj1RjltopA/hLvEl6pRSeQTYG0n6AsIqQi/8jaPbVgLg768TP4sqUEFpVEaGmzKXvBBGE4ihRouWlddmNYUKbEeS/ZYZlBlhNND30rKo1K+BQWPOO1KvKFDi8sY1sxMiJGZowqwf71MXLYfbpS/yOnF912MIZRF8oha7qI/+YyO2FoiPaBfPJbXW8lIf7ZIIuG6dBmu6hnURQZDReJqO7qEqR3CEFF6ffFuTgLr9YY4i0m+tupHmi5XTUrz3EfZTkNZOWIrJb6IuhaLL8dUWFKmjFwM3+mp8vt+eNIN8/NWttE0CH9dAIkC9VNyPT134CziOB5zn2k3XUeuyjFmQ7vNqDne9YCAkoTXB4ULhi7O/Dogps9CUQaYxpqtAGPMB2l9